# Astronomy Supervised

This example demonstrates a supervised astronomy workflow for classifying astronomical transients
from their multi-band light curves.

> **Note:** This notebook is intentionally slightly more involved than the [Hello World](getting_started.ipynb)
> and [Astronomy Unsupervised](unsupervised_image_extragalactic.ipynb) examples. In this example, we define a
> **custom dataset class** and a **custom model** for 1-D light-curve data. If you are new to Hyrax, we
> recommend working through those earlier examples first.

## Overview

The workflow is:

- Download the PLAsTiCC light-curve dataset from the Multi-Modal Universe project
- Preprocess the variable-length, irregularly sampled light curves into per-observation sequences
- Define a custom Hyrax dataset (with a custom `collate_*` function for variable-length padding) and a GRU-based recurrent model
- Train the classifier with class-weighted loss to handle PLAsTiCC's severe class imbalance
- Predict transient classes on held-out test data
- Evaluate performance with a confusion matrix

The [PLAsTiCC challenge](https://plasticc.org/) was a major Kaggle
competition for photometric transient classification on simulated LSST light curves.
The winning solution, [AVOCADO](https://arxiv.org/abs/1907.04690), used
Gaussian Process interpolation followed by a gradient-boosted tree. Several other top-placing
teams — including the 2nd-place solution — used recurrent neural networks (GRUs and LSTMs)
on the raw observation sequences.

We follow this RNN-based approach here because it maps naturally onto Hyrax's dataset and
model APIs and demonstrates how to handle **variable-length, irregularly sampled time series**
without any interpolation or binning. GRUs process observations sequentially and encode the
time gap between observations as an explicit input feature.

## The data

This example uses the [PLAsTiCC](https://plasticc.org/) training set as reformatted by the
[Multi-Modal Universe](https://github.com/MultimodalUniverse/MultimodalUniverse) (MMU) project.
It contains roughly 7,800 labeled light curves across 14 classes of astronomical transients
and variables — including SNIa, SNII, SNIbc, TDE, AGN, RR Lyrae, kilonova, and more.
Each light curve has multi-band photometry (LSST *ugrizY*) with irregular cadence.

## The model

We define a GRU (Gated Recurrent Unit) recurrent neural network that processes each light
curve as a variable-length sequence of observations. Each observation is represented as a
feature vector containing the time elapsed since the previous observation, the flux, the
flux uncertainty, and a one-hot encoding of the photometric band. This approach naturally
handles irregular cadence and variable-length sequences — unlike a CNN, the GRU does not
require binning or interpolation, so no temporal information is lost.

## Install dependencies

We need Hyrax and the HuggingFace `datasets` library to download the MMU data.
You can skip this step if these are already installed in your environment.

In [ ]:
# %pip install hyrax

## Download the Preprocessed PLAsTiCC dataset

We have preprocessed the MMU PLAsTiCC dataset into a format suitable for training a model.
The preprocessed data is available on Zenodo, and we can download it with `pooch` as shown below.

The total size for the train and test files is about 41 MB.
An additional json file contains a mapping from class index to transient name, which is useful for interpreting results.

In [ ]:
import pooch

train_file_path = pooch.retrieve(
    # DOI for preprocessed PLAsTiCC training dataset
    url="doi:10.5281/zenodo.20415708/train.npz",
    known_hash="md5:a33195f564484c3ec487eadd101b0a70",
    fname="train.npz",
    path="./data/plasticc",
)

test_file_path = pooch.retrieve(
    # DOI for preprocessed PLAsTiCC test dataset
    url="doi:10.5281/zenodo.20415708/test.npz",
    known_hash="md5:b0c0aade9f21f41705954c621d72fd18",
    fname="test.npz",
    path="./data/plasticc",
)

class_mapping_file_path = pooch.retrieve(
    # DOI for preprocessed PLAsTiCC class mapping
    url="doi:10.5281/zenodo.20415708/idx_to_name.json",
    known_hash="md5:ac5c2fa99b061369295e81922c9f9a4c",
    fname="idx_to_name.json",
    path="./data/plasticc",
)

## Define a custom dataset

To use preprocessed data with Hyrax we subclass `HyraxDataset`.
A dataset needs to implement:
- `__len__` — the number of samples
- `get_<field>(idx)` methods for each field the model will consume

Because our light-curve sequences have variable lengths, we also define a `collate_sequence`
function. Hyrax automatically detects this method and use it instead of the default
numpy-stacking collation. Our `collate_sequence` function pads all sequences in
a batch to the same length and returns the true sequence lengths.

In [ ]:
import numpy as np
from pathlib import Path
from hyrax.datasets import HyraxDataset


class PLAsTiCCSequenceDataset(HyraxDataset):
    """Hyrax dataset for variable-length PLAsTiCC light-curve sequences.
    The value for data_location should be the path to either train.npz or test.npz
    file."""

    def __init__(self, config: dict, data_location=None):
        data_location = Path(data_location)

        # `data` is a dict with keys "sequences", "labels", and "object_ids"
        data = np.load(data_location, allow_pickle=True)
        self.sequences = data["sequences"]  # object array of (seq_len, 9) arrays
        self.labels = data["labels"]
        self.object_ids = data["object_ids"]

        super().__init__(config)

    def get_sequence(self, idx):
        """Return the feature sequence for one object — shape (seq_len, 9)."""
        return np.array(self.sequences[idx], dtype=np.float32)

    def get_label(self, idx):
        """Return the integer class label."""
        return int(self.labels[idx])

    def get_object_id(self, idx):
        """Return a unique string identifier. 
        Note: Hyrax expects identifiers to be strings to simplify data handling."""
        return str(self.object_ids[idx])

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.labels)

    def collate_sequence(self, batch: list[dict]) -> dict:
        """Pad variable-length sequences to the max length in the batch.

        Example
        -------
        If the batch has 3 samples with sequence lengths 5, 8, and 6, it might look like this:
        ```
        batch = [
            {"sequence": array of shape (5, 9), "label": 0, ...},
            {"sequence": array of shape (8, 9), "label": 1, ...},
            {"sequence": array of shape (6, 9), "label": 2, ...}
        ]
        ```

        The collated sequence will be a (3, 8, 9) array where the first sample is
        padded with zeros in the last 3 time steps, the second sample is unchanged,
        and the third sample is padded with zeros in the last 2 time steps.
        The "lengths" array will be [5, 8, 6].
        ```
        result = {
            "sequence": array of shape (3, 8, 9),
            "lengths": array([5, 8, 6])}
        ```

        Parameters
        ----------
        batch
            List of dicts, each with keys like ``"sequence"`` and ``"label"``.

        Returns
        -------
        dict
            ``"sequence"`` — float32 array (batch, max_len, 9), zero-padded
            ``"lengths"`` — int64 array (batch,) with true sequence lengths
        """
        result = {}

        seqs = [b["sequence"] for b in batch]
        lengths = np.array([len(s) for s in seqs], dtype=np.int64)
        max_len = int(lengths.max())

        padded = np.zeros((len(seqs), max_len, seqs[0].shape[-1]), dtype=np.float32)
        for i, s in enumerate(seqs):
            padded[i, : len(s)] = s

        result["sequence"] = padded
        result["lengths"] = lengths

        return result

## Experiment with the dataset

We can begin to use some of the Hyrax functionality to explore the dataset and verify that our custom dataset class is working as expected.

In [ ]:
from hyrax import Hyrax

h = Hyrax()

data_request = {
    "exploration": {
        "plasticc": {
            "dataset_class": "PLAsTiCCSequenceDataset",
            "data_location": "./data/plasticc/train.npz",
            "primary_id_field": "object_id",
        }
    }
}

h.set_config("data_request", data_request)

ds = h.prepare()

In [ ]:
ds['exploration'][0]

In [ ]:
ff = np.load("./data/plasticc/train.npz", allow_pickle=True)
print(f"label: {ff['object_ids'][0]}")
print(f"sequence shape: {ff['sequences'][0].shape}")
print(f"object id: {ff['object_ids'][0]}")

## Define a custom model

We use a bidirectional GRU that processes the padded observation sequences using
PyTorch's `pack_padded_sequence` / `pad_packed_sequence` to efficiently skip
padding positions. The final hidden states from both directions are concatenated
and passed through a classification head.

The model also sets its own loss function (`self.criterion`) with class weights
computed from the training data. Hyrax's `@hyrax_model` decorator detects this
and uses it instead of loading a criterion from the config.

> **Note:** As with the dataset class, the model below uses `.get()` with inline
> defaults for brevity. In your own projects, define these defaults in a TOML
> config file and use direct `[]` access. See the
> [Configuration System](../configuration_system.rst) documentation.

In [ ]:
import torch
import torch.nn as nn
from hyrax.models import hyrax_model


@hyrax_model
class LightCurveGRU(nn.Module):
    """A bidirectional GRU for variable-length multi-band light-curve classification."""

    def __init__(self, config, data_sample=None):
        super().__init__()
        self.config = config
        model_config = config["model"]["LightCurveGRU"]

        input_dim = model_config["input_dim"]
        hidden_size = model_config["hidden_size"]
        num_layers = model_config["num_layers"]
        dropout = model_config["dropout"]
        num_classes = model_config["output_classes"]
        self.grad_clip = model_config["grad_clip"]

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # Bidirectional → 2x hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):

        sequences, _= x
        _, hidden = self.gru(sequences)

        # hidden shape: (num_layers * 2, batch, hidden_size)
        # Concatenate final forward and backward hidden states
        fwd = hidden[-2]  # last layer, forward
        bwd = hidden[-1]  # last layer, backward
        combined = torch.cat([fwd, bwd], dim=-1)  # (batch, hidden_size * 2)

        return self.classifier(combined)

    def train_batch(self, batch):
        _, labels = batch
        self.optimizer.zero_grad()
        outputs = self(batch)
        loss = self.criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), self.grad_clip)
        self.optimizer.step()
        return {"loss": loss.item()}

    def validate_batch(self, batch):
        _, labels = batch
        outputs = self(batch)
        loss = self.criterion(outputs, labels)
        return {"loss": loss.item()}

    def infer_batch(self, batch):
        return self(batch)

    @staticmethod
    def prepare_inputs(data_dict):
        """Convert the collated data into the format that the model required.

        Notes:
        - By convention, we return a tuple of values that will be unpacked by the
        model's `train_batch`, `validate_batch`, and `infer_batch` methods.
        - The prepare_inputs function is saved with the trained model and used
        during inference to ensure that raw data is processed in the same way as
        during training/validation. This is why we have a special `import` statement
        here. We want to make sure any necessary libraries are imported within this
        function.
        """

        import numpy as np

        data = data_dict["data"]
        sequence = np.asarray(data["sequence"], dtype=np.float32)
        label = np.asarray(data.get("label", np.zeros(len(sequence))), dtype=np.int64)
        return (sequence, label)

## Initialize Hyrax and configure

From here the workflow is the same as any other Hyrax example: create a `Hyrax` instance,
point it at our custom model and dataset, and run verbs.

In [ ]:
from hyrax import Hyrax

h = Hyrax()

h.set_config("model.name", "LightCurveGRU")

# Create the config section for our custom model. We pass the class weights
# so the model can build a weighted CrossEntropyLoss in its __init__.
NUM_BANDS = 6  # ugrizy bands
INPUT_DIM = 1 + 1 + 1 + NUM_BANDS  # delta_t, flux, flux_err, 6 band one-hots = 9
OUTPUT_CLASSES = 14  # number of transient classes in PLAsTiCC

h.config["model"]["LightCurveGRU"] = {
    "output_classes": OUTPUT_CLASSES,
    "input_dim": INPUT_DIM,
    "hidden_size": 256,
    "num_layers": 2,
    "dropout": 0.3,
    "grad_clip": 1.0,
}

# Adam is the standard optimizer for RNNs — SGD struggles with recurrent gradients
h.set_config("optimizer.name", "torch.optim.Adam")
h.config["torch.optim.Adam"] = {"lr": 1e-3}

h.set_config("data_loader.batch_size", 64)
h.set_config("train.epochs", 50)

## Define the dataset and train

We configure the `data_request` to use our custom `PLAsTiCCSequenceDataset` and request
the `sequence` and `label` fields. Note how the field names match the `get_sequence`, `get_label`,
and `collate_sequence` methods we defined above. Because our dataset defines a `collate_sequence`
method, Hyrax will automatically use it to pad variable-length sequences in each batch.

In [ ]:
data_request_definition = {
    "train": {
        "data": {
            "dataset_class": "PLAsTiCCSequenceDataset",
            "data_location": "./data/plasticc/train.npz",
            "fields": ["sequence", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
        },
    },
    "validate": {
        "data": {
            "dataset_class": "PLAsTiCCSequenceDataset",
            "data_location": "./data/plasticc/train.npz",
            "fields": ["sequence", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
        },
    },
}

h.set_config("data_request", data_request_definition)

trained_model = h.train()

## Predict with the model

We now classify the held-out test light curves. We'll specify that we want to load
the test.npz file with the PLAsTiCCSequenceDataset class.

In [ ]:
data_request_definition["infer"] = {
    "data": {
        "dataset_class": "PLAsTiCCSequenceDataset",
        "data_location": "./data/plasticc/test.npz",
        "fields": ["sequence"],
        "primary_id_field": "object_id",
    },
}

h.set_config("data_request", data_request_definition)

inference_results = h.infer()

## Evaluate the performance

The model outputs a 14-element vector per light curve. The index of the maximum value
is the predicted class. We compare against the true labels and display a confusion matrix.

We'll leverage the builtin functionality of Hyrax to open the inference results and original data
files in such that they are index-aligned. 

In [ ]:
data_request = {
    "evaluation": {
        "infer_results": {
            "dataset_class": "ResultDataset",
            "data_location": "/home/drew/code/hyrax/docs/pre_executed/hyrax_workshop/results/20260527-120726-infer-7JER",
            "primary_id_field": "object_id",
        },
        "input_data": {
            "dataset_class": "PLAsTiCCSequenceDataset",
            "data_location": "./data/plasticc/test.npz",
            "fields": ["label", "object_id"],
        }
    }
}
h.set_config("data_request", data_request)
ds = h.prepare()

In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

with open("./data/plasticc/idx_to_name.json", "r") as f:
    idx_to_name = json.load(f)

# Predicted classes
y_pred = [ds['evaluation'][i]['infer_results']['data'].argmax() for i in range(len(ds['evaluation']))]

# True labels from test split
y_true = [ds['evaluation'][i]['input_data']['label'] for i in range(len(ds['evaluation']))]

correct = sum(t == p for t, p in zip(y_true, y_pred))
print(f"Accuracy: {correct / len(y_true):.2%}")

class_names = [idx_to_name[str(i)] for i in range(len(idx_to_name))]

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, values_format="d")
plt.tight_layout()
plt.show()

The GRU achieves roughly **70%** accuracy on 14 classes; but the PLAsTiCC Kaggle competition winners achieved much higher performance by combining Gaussian Process interpolation, hand-crafted features, and gradient-boosted trees. The goal of this notebook is not to compete with those results, but to show how Hyrax's model and dataset APIs can be used to build a recurrent classifier from scratch on time-series data. 